### PHASE 1 — Data Loading & Exploration
In this phase, we load the datasets and check basic details.

In [3]:
import pandas as pd
import os

# Folder path where CSV files are stored
folder_path = 'C:/Users/Pranav/Python codes/datasets_transportation/csv_files'

# Load datasets
fact_trips = pd.read_csv(os.path.join(folder_path, 'fact_trips.csv'))
dim_city = pd.read_csv(os.path.join(folder_path, 'dim_city.csv'))
dim_date = pd.read_csv(os.path.join(folder_path, 'dim_date.csv'))

# View first 5 rows
print("Fact Trips Table:")
fact_trips.head()

Fact Trips Table:


,trip_id,date,city_id,passenger_type,distance_travelled(km),fare_amount,passenger_rating,driver_rating
0,TRPLUC240113d55de2fb,2024-01-13,UP01,repeated,11,158,5,5
1,TRPVAD240129a3b6dba8,2024-01-29,GJ02,repeated,7,74,5,5
2,TRPCOI240107a42430fb,2024-01-07,TN01,repeated,11,155,8,8
3,TRPKOC240325d7601389,2024-03-25,KL01,repeated,36,427,9,10
4,TRPVIS2406027be97166,2024-06-02,AP01,new,17,265,8,8


In [2]:
# Check missing values
print("Missing Values:")
fact_trips.isnull().sum()

Missing Values:


trip_id                   0
date                      0
city_id                   0
passenger_type            0
distance_travelled(km)    0
fare_amount               0
passenger_rating          0
driver_rating             0
dtype: int64

### PHASE 2 — Feature Engineering
In this phase, we merge tables and create new columns for analysis.

In [4]:
# Merge tables to add city and date details
fact_trips = pd.merge(fact_trips, dim_city[['city_id', 'city_name']], on='city_id', how='left')
fact_trips = pd.merge(fact_trips, dim_date[['date', 'day_type']], on='date', how='left')

# Create new column: fare per km
fact_trips['fare_per_km'] = fact_trips['fare_amount'] / fact_trips['distance_travelled(km)']

# Show before rounding
print("Before rounding:")
fact_trips.head()

Before rounding:


,trip_id,date,city_id,passenger_type,distance_travelled(km),fare_amount,passenger_rating,driver_rating,city_name,day_type,fare_per_km
0,TRPLUC240113d55de2fb,2024-01-13,UP01,repeated,11,158,5,5,Lucknow,Weekend,14.363636
1,TRPVAD240129a3b6dba8,2024-01-29,GJ02,repeated,7,74,5,5,Vadodara,Weekday,10.571429
2,TRPCOI240107a42430fb,2024-01-07,TN01,repeated,11,155,8,8,Coimbatore,Weekend,14.090909
3,TRPKOC240325d7601389,2024-03-25,KL01,repeated,36,427,9,10,Kochi,Weekday,11.861111
4,TRPVIS2406027be97166,2024-06-02,AP01,new,17,265,8,8,Visakhapatnam,Weekend,15.588235


In [5]:
# Round values
fact_trips = fact_trips.round(2)

# Show after rounding
print("After rounding:")
fact_trips.head()

After rounding:


,trip_id,date,city_id,passenger_type,distance_travelled(km),fare_amount,passenger_rating,driver_rating,city_name,day_type,fare_per_km
0,TRPLUC240113d55de2fb,2024-01-13,UP01,repeated,11,158,5,5,Lucknow,Weekend,14.36
1,TRPVAD240129a3b6dba8,2024-01-29,GJ02,repeated,7,74,5,5,Vadodara,Weekday,10.57
2,TRPCOI240107a42430fb,2024-01-07,TN01,repeated,11,155,8,8,Coimbatore,Weekend,14.09
3,TRPKOC240325d7601389,2024-03-25,KL01,repeated,36,427,9,10,Kochi,Weekday,11.86
4,TRPVIS2406027be97166,2024-06-02,AP01,new,17,265,8,8,Visakhapatnam,Weekend,15.59


### PHASE 3 — Analysis & Insights
Here we generate some basic business insights from the data.

🔹 City-wise Revenue

In [6]:
city_revenue = fact_trips.groupby('city_name')['fare_amount'].sum().sort_values(ascending=False)

print("Total Revenue by City:")
city_revenue

Total Revenue by City:


city_name
Jaipur           37207497
Kochi            16997596
Chandigarh       11058401
Lucknow           9463551
Visakhapatnam     8018282
Indore            7635228
Surat             6431599
Mysore            4054745
Vadodara          3797200
Coimbatore        3523992
Name: fare_amount, dtype: int64

🔹 Revenue Comparison: Weekday vs Weekend

In [7]:
day_type_revenue = fact_trips.groupby('day_type')['fare_amount'].sum()

print("# Total revenue generated on weekdays vs weekends")
day_type_revenue

# Total revenue generated on weekdays vs weekends


day_type
Weekday    47431603
Weekend    60756488
Name: fare_amount, dtype: int64

🔹 Average Ratings

In [9]:
city_ratings = fact_trips.groupby('city_name')['passenger_rating'].mean().sort_values(ascending=False)

print("Average Ratings by City:")
city_ratings.round(2)

Average Ratings by City:


city_name
Mysore           8.70
Jaipur           8.58
Kochi            8.52
Visakhapatnam    8.43
Chandigarh       7.98
Coimbatore       7.88
Indore           7.83
Vadodara         6.61
Lucknow          6.49
Surat            6.42
Name: passenger_rating, dtype: float64

🔹 Top 5 Cities by Driver Rating

In [10]:
top_driver_cities = fact_trips.groupby('city_name')['driver_rating'].mean().sort_values(ascending=False).head(5)

print("🔹 Top 5 Cities by Driver Rating:")
top_driver_cities.round(2)

🔹 Top 5 Cities by Driver Rating:


city_name
Kochi            8.99
Visakhapatnam    8.99
Jaipur           8.99
Mysore           8.98
Chandigarh       7.72
Name: driver_rating, dtype: float64

### FINAL PHASE — Export Data
Save the final processed dataset.

In [11]:
output_file = os.path.join(folder_path, 'GoodCabs_Analysis.csv')

fact_trips.to_csv(output_file, index=False)

print("Process completed. File saved successfully.")

Process completed. File saved successfully.
